In [1]:
# Clean Bloomberg histories mechanically and export company, macro, and availability tables.
# Preserve valid negative values and missing prelisting data; do not interpolate.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import FINANCIAL_FIELDS, MARKET_FIELD_NAMES as MARKET_FIELDS, NONNEGATIVE_FIELDS, ERROR_PATTERN, MISSING_TOKENS
from pathlib import Path
from numbers import Real
from collections import Counter
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook

ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks").is_dir() and (path / "data").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or a folder inside it.")
INPUT_DIR = project_paths(ROOT).processed
OUTPUT_DIR = INPUT_DIR 

# Equity, working capital, earnings, cash flows, capex, revenue, interest,
# depreciation, inflation, GDP growth, and interest rates retain source signs.
stats = Counter()


In [3]:
def clean_text(value):
    if value is None or pd.isna(value):
        return pd.NA
    text = " ".join(str(value).split())
    if text.upper() in MISSING_TOKENS or re.match(ERROR_PATTERN, text, flags=re.I):
        return pd.NA
    return text


def standardize_company_id(value):
    text = clean_text(value)
    if pd.isna(text):
        raise ValueError("Security_List contains a missing company_id.")
    text = text.upper()
    match = re.fullmatch(r"(SA|NB)_(\d+)", text)
    if not match:
        raise ValueError(f"Unrecognized company_id: {text!r}")
    return f"{match[1]}_{int(match[2])}"


def standardize_security(value):
    text = clean_text(value)
    if pd.isna(text):
        raise ValueError("Security_List contains a missing Bloomberg identifier.")
    parts = text.upper().split()
    if parts[-1] in {"EQUITY", "INDEX", "CORP"}:
        parts[-1] = parts[-1].title()
    return " ".join(parts)


def parse_dates(values):
    raw = pd.Series(values, dtype=object)
    numeric = raw.map(lambda value: isinstance(value, Real) and not isinstance(value, (bool, np.bool_)))
    result = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
    if numeric.any():
        serial = pd.to_numeric(raw.loc[numeric], errors="coerce")
        serial = serial.where(serial.between(1, 100000))
        result.loc[numeric] = pd.to_datetime(serial, unit="D", origin="1899-12-30", errors="coerce")
    if (~numeric).any():
        text = raw.loc[~numeric].map(clean_text)
        result.loc[~numeric] = pd.to_datetime(text, format="mixed", errors="coerce")
    return result.dt.normalize()


def clean_numeric_values(frame):
    frame = frame.copy()
    raw = frame.pop("raw_value")
    text = raw.astype("string").str.strip()
    upper = text.str.upper()
    missing = raw.isna() | upper.isin(MISSING_TOKENS)
    errors = upper.str.match(ERROR_PATTERN, na=False)
    # Remove grouping commas; convert accounting parentheses without changing units.
    numeric_text = text.str.replace(",", "", regex=False).str.replace(r"^\((.+)\)$", r"-\1", regex=True)
    value = pd.to_numeric(numeric_text.mask(missing | errors), errors="coerce").astype(float)
    boolean = raw.map(lambda item: isinstance(item, (bool, np.bool_)))
    value = value.mask(boolean)
    nonfinite = value.notna() & ~np.isfinite(value)
    negative = frame["field"].isin(NONNEGATIVE_FIELDS) & value.lt(0)
    nonpositive_price = frame["field"].eq("adjusted_price") & value.le(0)
    invalid_unemployment = frame["field"].eq("unemployment_rate") & (value.lt(0) | value.gt(100))
    impossible = nonfinite | negative | nonpositive_price | invalid_unemployment
    issue = pd.Series("", index=frame.index, dtype="string")
    issue.loc[~missing & ~errors & value.isna()] = "not_numeric"
    issue.loc[errors] = "bloomberg_error"
    issue.loc[nonfinite] = "nonfinite"
    issue.loc[negative] = "negative_nonnegative_field"
    issue.loc[nonpositive_price] = "nonpositive_price"
    issue.loc[invalid_unemployment] = "unemployment_outside_0_100"
    # Flag impossible values and leave them missing; original workbooks are untouched.
    frame["value"] = value.mask(impossible)
    frame["is_impossible"] = impossible.astype(bool)
    frame["value_issue"] = issue
    return frame


def deduplicate_records(frame, keys):
    if frame.empty or not frame.duplicated(keys).any():
        return frame.reset_index(drop=True)
    stats["duplicate_records_removed"] += int(frame.duplicated(keys).sum())
    duplicates = frame.loc[frame.duplicated(keys, keep=False)]
    conflict_keys = (
        duplicates.groupby(keys, sort=False)["value"].nunique(dropna=True)
        .loc[lambda counts: counts.gt(1)].index
    )
    # A usable value wins over a missing duplicate. Conflicting usable values
    # become missing instead of selecting a later observation or averaging.
    ordered = frame.assign(_missing=frame["value"].isna()).sort_values("_missing", kind="stable")
    result = ordered.drop_duplicates(keys).drop(columns="_missing").copy()
    if len(conflict_keys):
        conflicts = pd.MultiIndex.from_frame(result[keys]).isin(conflict_keys)
        result.loc[conflicts, "value"] = np.nan
        result.loc[conflicts, "value_issue"] = "conflicting_duplicate"
        stats["conflicting_duplicate_keys"] += len(conflict_keys)
    return result.reset_index(drop=True)


def parse_paired_sheet(sheet, company_id=None, start_date=None, end_date=None, prediction_date=None):
    rows = sheet.iter_rows(values_only=True)
    headers = [str(value).strip() if value is not None else "" for value in next(rows, ())]
    if len([header for header in headers if header]) != len(set(header for header in headers if header)):
        raise ValueError(f"Duplicate headers in {sheet.title}")
    pairs = []
    for index, header in enumerate(headers):
        if header.endswith("_Date"):
            field = header[:-5]
            value_header = f"{field}_Value"
            if value_header not in headers:
                raise ValueError(f"Missing {value_header} in {sheet.title}")
            if company_id is not None and field not in FINANCIAL_FIELDS | MARKET_FIELDS:
                raise ValueError(f"Unrecognized company field {field!r} in {sheet.title}")
            pairs.append((field, index, headers.index(value_header)))
        elif header.endswith("_Value") and f"{header[:-6]}_Date" not in headers:
            raise ValueError(f"Missing paired date column in {sheet.title}: {header}")
    if not pairs:
        raise ValueError(f"No paired Date/Value columns in {sheet.title}")
    records = []
    for row in rows:
        for field, di, vi in pairs:
            raw_date = row[di] if di < len(row) else None
            raw_value = row[vi] if vi < len(row) else None
            if raw_date is not None or raw_value is not None:
                records.append((raw_date, field, raw_value))
    frame = pd.DataFrame(records, columns=["date", "field", "raw_value"])
    frame["date"] = parse_dates(frame["date"])
    stats["invalid_date_records_removed"] += int(frame["date"].isna().sum())
    frame = frame.loc[frame["date"].notna()].reset_index(drop=True)
    in_window = pd.Series(True, index=frame.index)
    if start_date is not None:
        in_window &= frame["date"].ge(start_date)
    if end_date is not None:
        in_window &= frame["date"].le(end_date)
    if prediction_date is not None and pd.notna(prediction_date):
        in_window &= frame["date"].lt(prediction_date)
    stats["outside_window_or_prediction_records_removed"] += int((~in_window).sum())
    frame = clean_numeric_values(frame.loc[in_window].reset_index(drop=True))
    stats["impossible_values"] += int(frame["is_impossible"].sum())
    stats["bloomberg_errors"] += int(frame["value_issue"].eq("bloomberg_error").sum())
    if company_id is not None:
        frame.insert(0, "company_id", company_id)
        frame.insert(3, "data_type", np.where(frame["field"].isin(FINANCIAL_FIELDS), "financial", "market"))
    return frame


def read_security_list(sheet):
    rows = sheet.iter_rows(values_only=True)
    headers = [str(value).strip() if value is not None else "" for value in next(rows, ())]
    required = {"company_id", "company_type", "bloomberg_identifier", "start_date", "end_date", "bloomberg_sector", "bloomberg_industry"}
    if not required.issubset(headers):
        raise ValueError(f"Security_List is missing columns: {sorted(required - set(headers))}")
    if len(headers) != len(set(headers)):
        raise ValueError("Security_List contains duplicate headers.")
    metadata = pd.DataFrame(list(rows), columns=headers).dropna(how="all")
    metadata["company_id"] = metadata["company_id"].map(standardize_company_id)
    metadata["bloomberg_identifier"] = metadata["bloomberg_identifier"].map(standardize_security)
    metadata["company_type"] = metadata["company_type"].map(clean_text).astype("string").str.lower()
    if not metadata["company_type"].isin(["bankrupt", "nonbankrupt"]).all():
        raise ValueError("Unknown company_type in Security_List.")
    for column in ["name", "ticker", "Sector", "Industry", "bloomberg_sector", "bloomberg_industry"]:
        if column in metadata:
            metadata[column] = metadata[column].map(clean_text).astype("string")
    if "ticker" in metadata:
        metadata["ticker"] = metadata["ticker"].str.upper()
    for column in ["start_date", "end_date", "prediction_date", "bankruptcy_date"]:
        if column not in metadata:
            metadata[column] = pd.NaT
        else:
            raw = metadata[column]
            parsed = parse_dates(raw)
            if column == "prediction_date" and (raw.map(clean_text).notna() & parsed.isna()).any():
                raise ValueError("A supplied prediction_date is invalid; refusing to remove its cutoff.")
            metadata[column] = parsed.to_numpy()
    if metadata[["start_date", "end_date"]].isna().any().any() or metadata["start_date"].gt(metadata["end_date"]).any():
        raise ValueError("Missing or invalid collection windows in Security_List.")
    if metadata.loc[metadata["company_type"].eq("bankrupt"), "prediction_date"].isna().any():
        raise ValueError("Bankrupt companies require prediction_date for a historical cutoff.")
    return metadata.reset_index(drop=True)


In [4]:
# Read the populated Security_List metadata, not the unpopulated CSV classifications.
# Missing either Bloomberg sector or industry excludes the entire company.
workbook_paths = sorted(
    path for path in INPUT_DIR.rglob("*")
    if path.suffix.lower() in {".xlsx", ".xlsm"} and not path.name.startswith("~$")
)
if not workbook_paths:
    raise FileNotFoundError(f"No Bloomberg workbooks under {INPUT_DIR}")
company_chunks = []
metadata_chunks = []
macro_sources = []
for path in workbook_paths:
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        if "Macro_Data" in workbook.sheetnames:
            macro_sources.append(path)
        if "Security_List" not in workbook.sheetnames:
            continue
        local = read_security_list(workbook["Security_List"])
        stats["input_company_records"] += len(local)
        missing_classification = local[["bloomberg_sector", "bloomberg_industry"]].isna().any(axis=1)
        # Explicit unknown placeholders are not usable Bloomberg classifications.
        missing_classification |= local[["bloomberg_sector", "bloomberg_industry"]].apply(
            lambda column: column.str.upper().isin(["UNKNOWN", "NOT_FOUND", "0"])
        ).any(axis=1)
        stats["company_records_missing_classification"] += int(missing_classification.sum())
        all_ids = set(local["company_id"])
        local = local.loc[~missing_classification].copy()
        metadata_chunks.append(local)
        by_id = local.set_index("company_id")
        if not by_id.index.is_unique:
            raise ValueError(f"Duplicate standardized company IDs in {path.name}")
        found = set()
        for sheet in workbook:
            if sheet.title in {"Security_List", "Macro_Data"}:
                continue
            match = re.match(r"^((?:SA|NB)_\d+)(?:_|$)", sheet.title, flags=re.I)
            if not match:
                raise ValueError(f"Cannot map worksheet {sheet.title!r} in {path.name}")
            company_id = standardize_company_id(match[1])
            if company_id not in all_ids:
                raise ValueError(f"Worksheet {sheet.title!r} is absent from Security_List in {path.name}")
            if company_id not in by_id.index:
                continue
            record = by_id.loc[company_id]
            frame = parse_paired_sheet(sheet, company_id, record["start_date"], record["end_date"], record["prediction_date"])
            company_chunks.append(frame)
            found.add(company_id)
        stats["company_records_missing_sheet"] += len(set(by_id.index) - found)
    finally:
        workbook.close()
if not metadata_chunks:
    raise ValueError("No Security_List worksheets were found.")
metadata = pd.concat(metadata_chunks, ignore_index=True)
# Identical repeated metadata can be collapsed. Conflicting metadata must be
# resolved in the inputs instead of silently choosing a different date window.
metadata = metadata.drop_duplicates()
if metadata["company_id"].duplicated().any():
    raise ValueError("Conflicting Security_List metadata for a company_id across workbooks.")
metadata = metadata.set_index("company_id", drop=False)
if not company_chunks:
    raise ValueError("No company histories have usable Bloomberg classifications.")
clean_company_long = deduplicate_records(
    pd.concat(company_chunks, ignore_index=True), ["company_id", "date", "field"]
)
del company_chunks

# Exclude firms without even one valid total-assets observation inside their window.
asset_ids = set(clean_company_long.loc[
    clean_company_long["field"].eq("total_assets") & clean_company_long["value"].notna(), "company_id"
])
stats["classified_companies_without_assets"] = len(set(metadata.index) - asset_ids)
metadata = metadata.loc[metadata.index.isin(asset_ids)].copy()
clean_company_long = clean_company_long.loc[clean_company_long["company_id"].isin(asset_ids)].copy()
if clean_company_long.empty:
    raise ValueError("No companies remain after classification and total-assets checks.")
clean_company_long = clean_company_long.sort_values(["company_id", "date", "field"]).reset_index(drop=True)

if not macro_sources:
    raise FileNotFoundError("No Macro_Data worksheet found under data/processed.")
macro_chunks = []
macro_start = metadata["start_date"].min()
macro_end = metadata["end_date"].max()
for path in macro_sources:
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        macro_chunks.append(parse_paired_sheet(workbook["Macro_Data"], start_date=macro_start, end_date=macro_end))
    finally:
        workbook.close()
clean_macro_long = deduplicate_records(pd.concat(macro_chunks, ignore_index=True), ["date", "field"])
clean_macro_long = clean_macro_long.sort_values(["date", "field"]).reset_index(drop=True)
del macro_chunks


In [5]:
# Availability uses valid assets and prices, as in the requested example.
# History lengths count distinct observed calendar years/months, not elapsed
# spans: missing years/months and unavailable pre-IPO periods never count.
def get_availability_dates(company_df):
    financial_dates = company_df.loc[
        company_df["field"].eq("total_assets") & company_df["value"].notna(), "date"
    ]
    market_dates = company_df.loc[
        company_df["field"].eq("adjusted_price") & company_df["value"].notna(), "date"
    ]
    observed_dates = company_df.loc[company_df["value"].notna(), "date"]
    return {
        "first_financial_date": financial_dates.min(),
        "last_financial_date": financial_dates.max(),
        "first_market_date": market_dates.min(),
        "last_market_date": market_dates.max(),
        "first_observed_date": observed_dates.min(),
        "last_observed_date": observed_dates.max(),
        "financial_history_years": financial_dates.dt.year.nunique(),
        "market_history_months": market_dates.dt.to_period("M").nunique(),
    }

availability = pd.DataFrame([
    {"company_id": company_id, **get_availability_dates(frame)}
    for company_id, frame in clean_company_long.groupby("company_id", sort=True)
])
company_availability = metadata.reset_index(drop=True).merge(
    availability, on="company_id", how="inner", validate="one_to_one"
).sort_values("company_id").reset_index(drop=True)
# Metadata/classifications live here; long histories join on company_id.
# Do not forward-fill field values or infer pseudo-dates from collection bounds.


In [6]:
# Validate and export only the three requested tables.
limits = company_availability.set_index("company_id")
cutoffs = clean_company_long["company_id"].map(limits["prediction_date"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for filename, frame in [
    ("clean_company_long.csv", clean_company_long),
    ("clean_macro_long.csv", clean_macro_long),
    ("company_availability.csv", company_availability),
]:
    frame.to_csv(OUTPUT_DIR / filename, index=False, date_format="%Y-%m-%d")
